# Projekt 1: Analiza wielowymiarowa potencjału turystycznego regionów bałkańskich
**Autorzy:** [Wpisz imiona i nazwiska / Podział zadań]

**Cel pracy:** 
Celem projektu jest wielowymiarowa ocena atrakcyjności turystycznej wybranych regionów na Bałkanach (poziom NUTS 2). Do analizy wykorzystano techniki porządkowania liniowego w celu stworzenia rankingu regionów oraz analizę skupień do wyodrębnienia grup o podobnej charakterystyce.


In [ ]:
# Import niezbędnych bibliotek
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")


## 1. Wczytanie danych, inżynieria cech i standaryzacja
Aby badanie miało charakter przestrzenny, skupiamy się na danych przekrojowych z najnowszego dostępnego roku (np. 2024). Na etapie inżynierii cech tworzymy **wskaźnik zagraniczności noclegów**, co pozwala usunąć wysoce współliniową kolumnę wartości bezwzględnych.

Standaryzacja danych (Z-score) sprowadza cechy o różnych rzędach wielkości do wspólnej skali, co jest krytycznym wymogiem przed analizą skupień oraz metodami porządkowania liniowego.


In [ ]:
# Wczytanie danych
df_raw = pd.read_excel('baza_balkany.xlsx')

# Filtrujemy dane dla najnowszego roku
latest_year = df_raw['Year'].max()
df = df_raw[df_raw['Year'] == latest_year].copy()
df.set_index('geo', inplace=True)

# Tworzenie wskaźnika zagraniczności noclegów
df['Wskaznik_Zagranicznosci'] = df['Noclegi_Zagraniczne'] / df['Noclegi_Ogolem']

# Usunięcie surowej kolumny Noclegi_Zagraniczne w celu uniknięcia dublowania wymiarów (współliniowości)
df = df.drop(columns=['Noclegi_Zagraniczne'])

# Definiujemy zmienne do analizy (nowy wskaźnik traktujemy jako stymulantę)
stymulanty = ['Noclegi_Ogolem', 'Liczba_Obiektow', 'Ruch_Lotniczy', 'Wody_Doskonała_Jakość_NUTS0', 'Wskaznik_Zagranicznosci']
destymulanty = ['Poziom_Cen_NUTS0', 'Robbery'] # Jeśli "Robbery" było pobrane w innym pliku, upewnij się że kolumna tak się nazywa
wybrane_zmienne = stymulanty + destymulanty

# Bezpieczne wyodrębnienie tylko tych zmiennych, które faktycznie istnieją w DF
dostepne_zmienne = [col for col in wybrane_zmienne if col in df.columns]
dostepne_destymulanty = [col for col in destymulanty if col in df.columns]

X = df[dostepne_zmienne].copy()

# Standaryzacja danych (Z-score)
scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)

print("Kształt danych po przygotowaniu:", X_std.shape)
display(X_std.head())


## 2. Analiza Skupień (Grupowanie)

Etapy grupowania:
1. Obliczenie indeksu Calińskiego-Harabasza w celu numerycznego potwierdzenia optymalnej liczby klastrów.
2. **Metoda hierarchiczna (Warda)**: oparta na minimalizacji wariancji wewnątrz nowo powstających skupień (wspomagana dendrogramem).
3. **Metoda podziałowa (k-średnich)**: klasyczna metoda algorytmiczna dla zweryfikowania stabilności podziału.


In [ ]:
# --- OCENA LICZBY KLASTRÓW (Indeks Calińskiego-Harabasza) ---
ch_scores = []
k_range = range(2, 10)

for k in k_range:
    kmeans_eval = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans_eval.fit_predict(X_std)
    ch_scores.append(calinski_harabasz_score(X_std, labels))

plt.figure(figsize=(8, 4))
plt.plot(k_range, ch_scores, marker='o', linestyle='--', color='teal')
plt.title('Indeks Calińskiego-Harabasza w celu doboru optymalnego k')
plt.xlabel('Liczba klastrów (k)')
plt.ylabel('Wartość indeksu CH (wyższa = lepsza)')
plt.show()

# Ustalamy liczbę klastrów na podstawie wykresu CH
k_clusters = 3

# --- 1. METODA WARDA I DENDROGRAM ---
Z_ward = linkage(X_std, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(Z_ward, labels=X_std.index, leaf_rotation=90)
plt.title('Dendrogram - Metoda Warda')
plt.xlabel('Regiony (NUTS)')
plt.ylabel('Odległość wiązania (Variance)')
plt.show()

# Przypisanie etykiet z metody Warda
df['Cluster_Ward'] = fcluster(Z_ward, k_clusters, criterion='maxclust')

# --- 2. METODA K-ŚREDNICH ---
kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10)
df['Cluster_KMeans'] = kmeans.fit_predict(X_std) + 1 

# Porównanie wyników grupowania
print("\nMacierz pomyłek obu metod grupowania:")
display(pd.crosstab(df['Cluster_Ward'], df['Cluster_KMeans'], 
                    rownames=['Ward'], colnames=['K-Means']))


## 3. Porządkowanie liniowe (Ranking)
Przeprowadzimy ocenę regionów tworząc syntetyczny miernik atrakcyjności:
1. **Metoda wzorcowa (Hellwiga)**: wykorzystuje odległość euklidesową od abstrakcyjnego punktu optymalnego ("wzorca rozwoju").
2. **Metoda bezwzorcowa**: klasyczna arytmetyczna agregacja zestawionych cech. Destymulanty zostały uprzednio przekształcone na stymulanty poprzez odwrócenie znaku.


In [ ]:
# --- 1. METODA HELLWIGA (Wzorcowa) ---
# Wzorzec rozwoju: maksimum dla stymulant, minimum dla destymulant
z_ideal_hellwig = X_std.max()
z_ideal_hellwig[dostepne_destymulanty] = X_std[dostepne_destymulanty].min()

# Obliczenie odległości
d_hellwig = np.sqrt(((X_std - z_ideal_hellwig)**2).sum(axis=1))
d0 = d_hellwig.mean() + 2 * d_hellwig.std(ddof=1)
df['Hellwig_Score'] = 1 - (d_hellwig / d0)

# --- 2. METODA BEZWZORCOWA (Suma zestandaryzowana) ---
X_bez = X_std.copy()
# Destymulanty zamieniamy na stymulanty
X_bez[dostepne_destymulanty] = -X_bez[dostepne_destymulanty] 
df['Bezwzorcowa_Score'] = X_bez.sum(axis=1) / len(dostepne_zmienne)

# Tworzenie rankingów
df['Rank_Hellwig'] = df['Hellwig_Score'].rank(ascending=False).astype(int)
df['Rank_Bezwzorcowa'] = df['Bezwzorcowa_Score'].rank(ascending=False).astype(int)

# Podgląd Top 10 regionów (według metody Hellwiga)
ranking_cols = ['Hellwig_Score', 'Rank_Hellwig', 'Bezwzorcowa_Score', 'Rank_Bezwzorcowa']
print("\nTop 10 najbardziej atrakcyjnych turystycznie regionów Bałkanów:")
display(df[ranking_cols].sort_values('Rank_Hellwig').head(10))


## 4. Podsumowanie i interpretacja wyników
Na koniec sprawdzimy zgodność zastosowanych metod porządkowania (współczynnik Spearmana) oraz przeanalizujemy profile powstałych klastrów.


In [ ]:
# Korelacja wyników z metod porządkowania
korelacja_rankingow = df[['Rank_Hellwig', 'Rank_Bezwzorcowa']].corr(method='spearman')
print("Korelacja rang Spearmana między metodą wzorcową i bezwzorcową:")
display(korelacja_rankingow)

# Profilowanie klastrów
agg_dict = {
    'Hellwig_Score': 'mean',
    'Bezwzorcowa_Score': 'mean',
    'Noclegi_Ogolem': 'mean',
    'Liczba_Obiektow': 'mean',
    'Wskaznik_Zagranicznosci': 'mean'
}

# Zabezpieczenie na wypadek brakujących kolumn
agg_dict = {k: v for k, v in agg_dict.items() if k in df.columns}

cluster_summary = df.groupby('Cluster_Ward').agg(agg_dict).round(3)

print("\nCharakterystyka klastrów (Metoda Warda) w odniesieniu do wskaźników i rankingu:")
display(cluster_summary.sort_values(by='Hellwig_Score', ascending=False))

# Wykres rozrzutu dla obu metod z kolorowaniem po klastrach
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='Hellwig_Score', 
    y='Bezwzorcowa_Score', 
    hue='Cluster_Ward', 
    palette='viridis', 
    data=df, 
    s=120, 
    edgecolor='black'
)
plt.title('Zależność między miernikami rozwoju a przypisaniem do klastrów')
plt.xlabel('Wskaźnik Hellwiga (Metoda Wzorcowa)')
plt.ylabel('Wskaźnik Bezwzorcowy')
plt.legend(title='Klaster (Ward)')
plt.show()


In [ ]:
# Zapis zintegrowanych wyników do pliku w celu ewentualnego raportowania w QGIS/Excelu
df.to_csv('Balkany_Wyniki_Taksonomia.csv')
print("Eksport bazy z rankingami i klastrami zakończony sukcesem (Balkany_Wyniki_Taksonomia.csv).")
